In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler


file_path = r'..\data\Indian_Stocks_10Yrs_Prices_By_Sector.xlsx'

df_bfsi = pd.read_excel(file_path, sheet_name='BFSI')
df_it = pd.read_excel(file_path, sheet_name='IT')
df_pharma = pd.read_excel(file_path, sheet_name='PHARMA')

# Merge on Date
df = df_bfsi.merge(df_it, on='Date', how='outer').merge(df_pharma, on='Date', how='outer')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

df.to_csv('merged_data.csv')

In [4]:
df=pd.read_csv('merged_data.csv', index_col='Date', parse_dates=True)

In [5]:
df.isna().sum()

HDFCBANK         0
ICICIBANK        0
SBIN             0
LICI          1549
BAJFINANCE       0
KOTAKBANK        0
AXISBANK         0
SBILIFE        409
HDFCLIFE       442
ICICIPRULI     160
TCS              0
INFY             0
HCLTECH          1
WIPRO            0
TECHM            0
LTIM           114
PERSISTENT       1
MPHASIS          1
COFORGE          0
OFSS             1
SUNPHARMA        1
DIVISLAB         1
DRREDDY          1
CIPLA            1
LUPIN            1
AUROPHARMA       2
ZYDUSLIFE        1
TORNTPHARM       1
ALKEM            2
BIOCON           1
dtype: int64

In [6]:
threshold = len(df) * 0.7
df = df.dropna(thresh=threshold, axis=1)

In [7]:
df.isna().sum()

HDFCBANK        0
ICICIBANK       0
SBIN            0
BAJFINANCE      0
KOTAKBANK       0
AXISBANK        0
SBILIFE       409
HDFCLIFE      442
ICICIPRULI    160
TCS             0
INFY            0
HCLTECH         1
WIPRO           0
TECHM           0
LTIM          114
PERSISTENT      1
MPHASIS         1
COFORGE         0
OFSS            1
SUNPHARMA       1
DIVISLAB        1
DRREDDY         1
CIPLA           1
LUPIN           1
AUROPHARMA      2
ZYDUSLIFE       1
TORNTPHARM      1
ALKEM           2
BIOCON          1
dtype: int64

In [8]:
df = df.ffill().bfill()
returns = df.pct_change().dropna()
scaler = StandardScaler()
scaled_returns = scaler.fit_transform(returns)

In [9]:
data_tensor = torch.tensor(scaled_returns.T, dtype=torch.float32)
stock_names = list(returns.columns)
num_stocks = len(stock_names)

In [10]:
class GraphDiscoveryNet(nn.Module):
    def __init__(self, num_nodes, lookback):
        super(GraphDiscoveryNet, self).__init__()
        # This is the "Discovery" part: A learnable weight matrix for all possible pairs
        self.adj_weights = nn.Parameter(torch.randn(num_nodes, num_nodes))
        
        self.gcn_layer = nn.Linear(lookback, 64)
        self.output_layer = nn.Linear(64, 1) # Predict next-day return

    def forward(self, x):
        # Apply sigmoid/softmax to weights to keep them normalized
        adj = torch.sigmoid(self.adj_weights)
        
        # Message Passing
        # x shape: [num_stocks, lookback]
        support = self.gcn_layer(x)
        output = torch.matmul(adj, support)
        prediction = self.output_layer(output)
        
        return prediction, adj

# Hyperparameters
lookback = 20
model = GraphDiscoveryNet(num_stocks, lookback)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.MSELoss()

# 4. TRAINING LOOP
# Preparing sliding window data
X_train = []
Y_train = []
for i in range(data_tensor.shape[1] - lookback - 1):
    X_train.append(data_tensor[:, i:i+lookback])
    Y_train.append(data_tensor[:, i+lookback])

for epoch in range(100):
    total_loss = 0
    for i in range(len(X_train)):
        optimizer.zero_grad()
        pred, learned_adj = model(X_train[i])
        loss = criterion(pred.squeeze(), Y_train[i])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {total_loss/len(X_train):.6f}")

Epoch 0 | Loss: 4.283424
Epoch 10 | Loss: 1.009775
Epoch 20 | Loss: 0.990092
Epoch 30 | Loss: 0.989928
Epoch 40 | Loss: 0.988968
Epoch 50 | Loss: 0.988209
Epoch 60 | Loss: 0.987602
Epoch 70 | Loss: 0.987355
Epoch 80 | Loss: 0.987053
Epoch 90 | Loss: 0.986600


In [11]:
# Final Learned Matrix
with torch.no_grad():
    _, final_adj = model(X_train[-1])
    adj_matrix = final_adj.cpu().numpy()

# Extract top relationships
pairs = []
for i in range(num_stocks):
    for j in range(num_stocks):
        if i != j:
            pairs.append({
                'Stock_A': stock_names[i],
                'Stock_B': stock_names[j],
                'Strength': adj_matrix[i, j]
            })

pairs_df = pd.DataFrame(pairs).sort_values(by='Strength', ascending=False)

# CATEGORIZE FOR RL TEAM
# Top 10% are Positive Relationship Pairs
positive_pairs = pairs_df.head(20) 
# Bottom 10% (or inverse patterns) are Negative Relationship Pairs
negative_pairs = pairs_df.tail(20)

# Save to CSV for the platform/RL team
positive_pairs.to_csv('discovered_positive_pairs.csv', index=False)
negative_pairs.to_csv('discovered_negative_pairs.csv', index=False)

print("Discovery Complete. Pairs saved for RL Team.")

Discovery Complete. Pairs saved for RL Team.


In [12]:
import os
import torch

# '../models' moves up one level from /notebooks to the parent, then into /models
model_dir = os.path.join('..', 'models')
model_name = 'gnn_pairs_discovery1.pth'
model_path = os.path.join(model_dir, model_name)

# 1. Create the 'models' directory if it doesn't exist
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"Created directory: {os.path.abspath(model_dir)}")

# 2. Save the model weights
try:
    torch.save(model.state_dict(), model_path)
    print(f"✅ Model saved successfully at: {model_path}")
except Exception as e:
    print(f"❌ Failed to save model: {e}")

✅ Model saved successfully at: ..\models\gnn_pairs_discovery1.pth


In [20]:
# 1. Check how many stocks the CSV has right now
print(f"Current stocks in CSV: {len(returns.columns)}")
print(f"Stock list: {list(returns.columns)}")

# 2. Check what the saved model expects
checkpoint = torch.load(model_path, map_location=device)
saved_size = checkpoint['adj_weights'].shape[0]
print(f"Saved model expects: {saved_size} stocks")

Current stocks in CSV: 30
Stock list: ['HDFCBANK', 'ICICIBANK', 'SBIN', 'LICI', 'BAJFINANCE', 'KOTAKBANK', 'AXISBANK', 'SBILIFE', 'HDFCLIFE', 'ICICIPRULI', 'TCS', 'INFY', 'HCLTECH', 'WIPRO', 'TECHM', 'LTIM', 'PERSISTENT', 'MPHASIS', 'COFORGE', 'OFSS', 'SUNPHARMA', 'DIVISLAB', 'DRREDDY', 'CIPLA', 'LUPIN', 'AUROPHARMA', 'ZYDUSLIFE', 'TORNTPHARM', 'ALKEM', 'BIOCON']
Saved model expects: 29 stocks


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# 1. DEFINE ARCHITECTURE (Must match your training snippet exactly)
class GraphDiscoveryNet(nn.Module):
    def __init__(self, num_nodes, lookback):
        super(GraphDiscoveryNet, self).__init__()
        self.adj_weights = nn.Parameter(torch.randn(num_nodes, num_nodes))
        self.gcn_layer = nn.Linear(lookback, 64)
        self.output_layer = nn.Linear(64, 1)

    def forward(self, x):
        # Using sigmoid as per your provided training code
        adj = torch.sigmoid(self.adj_weights)
        support = self.gcn_layer(x)
        output = torch.matmul(adj, support)
        prediction = self.output_layer(output)
        return prediction, adj

# 2. SETUP PATHS & DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csv_path = 'merged_data.csv'
model_path = os.path.join('..', 'models', 'gnn_pairs_discovery1.pth')

# 3. REPLICATE TRAINING PREPROCESSING
df = pd.read_csv(csv_path, index_col='Date', parse_dates=True)

# Important: Dropping LICI to reach the 29 stocks your model expects
if 'LICI' in df.columns:
    df = df.drop(columns=['LICI'])

# Replicate: ffill -> bfill -> pct_change -> scale
df = df.ffill().bfill()
returns = df.pct_change().dropna()

# Fit the scaler exactly as you did during training
scaler = StandardScaler()
scaled_returns = scaler.fit_transform(returns)
data_tensor = torch.tensor(scaled_returns.T, dtype=torch.float32).to(device)

num_stocks = len(returns.columns) # Should be 29
lookback = 20

# 4. LOAD THE MODEL
model = GraphDiscoveryNet(num_stocks, lookback).to(device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Weights loaded. Stock count: {num_stocks}")

    # 5. RUN PREDICTIONS
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for i in range(data_tensor.shape[1] - lookback):
            x = data_tensor[:, i:i+lookback]
            y = data_tensor[:, i+lookback]
            
            pred, _ = model(x)
            all_preds.append(pred.squeeze().cpu().numpy())
            all_actuals.append(y.cpu().numpy())

    # 6. CALCULATE ACCURATE METRICS
    all_preds = np.array(all_preds)
    all_actuals = np.array(all_actuals)

    # MSE on scaled data (should be small, likely < 1.5)
    mse = mean_squared_error(all_actuals, all_preds)
    
    # Hit Ratio: Did the predicted sign match the actual sign?
    hit_ratio = np.mean(np.sign(all_preds) == np.sign(all_actuals))

    print("\n" + "="*30)
    print("      ACCURATE GNN METRICS      ")
    print("="*30)
    print(f"Mean Squared Error: {mse:.6f}")
    print(f"Directional Hit Ratio: {hit_ratio:.2%}")
    print("="*30)
else:
    print(f"Model file not found at {model_path}")

✅ Weights loaded. Stock count: 29

      ACCURATE GNN METRICS      
Mean Squared Error: 0.983416
Directional Hit Ratio: 52.45%
